# Queues & Deques in Python — FIFO Architecture & Algorithmic Patterns

> **Topic:** Queues & Deques | **Folder:** Data Structures & Algorithms

A **Queue** is a linear data structure following the **First-In, First-Out (FIFO)** principle.
The element inserted first is the first one to be removed.

---

## Table of Contents
1. [Queue Concepts & Operations](#1.-Queue-Concepts-&-Operations)
2. [Why `list.pop(0)` is an $O(n)$ Anti-Pattern](#2.-Why-`list.pop(0)`-is-an-$O(n)$-Anti-Pattern)
3. [`collections.deque` (Double-Ended Queue)](#3.-`collections.deque`-(Double-Ended-Queue))
4. [Thread-Safe Queues (`queue.Queue`)](#4.-Thread-Safe-Queues-(queue.Queue))
5. [Algorithmic Pattern 1: Breadth-First Search (BFS)](#5.-Algorithmic-Pattern-1:-Breadth-First-Search-(BFS))
6. [Algorithmic Pattern 2: Sliding Window Maximum (Monotonic Deque)](#6.-Algorithmic-Pattern-2:-Sliding-Window-Maximum-(Monotonic-Deque))
7. [Algorithmic Pattern 3: Circular Queue Implementation](#7.-Algorithmic-Pattern-3:-Circular-Queue-Implementation)
8. [Quick Reference Card](#8.-Quick-Reference-Card)


---
## 1. Queue Concepts & Operations

| Operation | Method | Time Complexity | Space Complexity |
|-----------|--------|-----------------|------------------|
| **Enqueue** | Add item to rear | $O(1)$ | $O(1)$ |
| **Dequeue** | Remove item from front | $O(1)$ | $O(1)$ |
| **Front / Peek** | Inspect front item | $O(1)$ | $O(1)$ |


---
## 2. Why `list.pop(0)` is an $O(n)$ Anti-Pattern

Using a Python list as a queue by calling `list.pop(0)` forces CPython to **shift all remaining $n-1$ elements left in memory**,
resulting in an $O(n)$ time complexity for every single dequeue operation!


In [ ]:
# Benchmarking list.pop(0) vs deque.popleft()
from collections import deque
import timeit

N = 50_000
t_list  = timeit.timeit(lambda: [list(range(N)).pop(0) for _ in range(1000)], number=1)
t_deque = timeit.timeit(lambda: [deque(range(N)).popleft() for _ in range(1000)], number=1)

print(f"list.pop(0) time   : {t_list:.4f} seconds")
print(f"deque.popleft() time: {t_deque:.4f} seconds")
print(f"deque is ~{t_list / t_deque:.1f}x faster!")


---
## 3. `collections.deque` (Double-Ended Queue)


In [ ]:
# Operating on both ends with collections.deque
dq = deque([10, 20, 30])
dq.append(40)        # Add right
dq.appendleft(0)     # Add left
print("Deque:", dq)

print("pop():", dq.pop())          # Remove right -> 40
print("popleft():", dq.popleft())  # Remove left  -> 0
print("Remaining:", dq)


---
## 4. Thread-Safe Queues (`queue.Queue`)


In [ ]:
import queue
import threading

q = queue.Queue()
q.put("Task 1")
q.put("Task 2")

print("Fetched from Thread Queue:", q.get())
q.task_done()


---
## 5. Algorithmic Pattern 1: Breadth-First Search (BFS)


In [ ]:
# Level-Order Traversal of Binary Tree using Queue
class TreeNode:
    def __init__(self, val=0, left=None, right=None):
        self.val = val
        self.left = left
        self.right = right

def level_order_traversal(root):
    if not root: return []
    res = []
    q = deque([root])
    
    while q:
        level_size = len(q)
        current_level = []
        for _ in range(level_size):
            node = q.popleft()
            current_level.append(node.val)
            if node.left: q.append(node.left)
            if node.right: q.append(node.right)
        res.append(current_level)
        
    return res

# Tree: 1 -> (2, 3 -> (4, 5))
root = TreeNode(1, TreeNode(2), TreeNode(3, TreeNode(4), TreeNode(5)))
print("Level Order Traversal:", level_order_traversal(root))


---
## 6. Algorithmic Pattern 2: Sliding Window Maximum (Monotonic Deque)


In [ ]:
# Sliding Window Maximum in O(n) using Monotonic Deque
def max_sliding_window(nums, k):
    dq = deque()  # Stores indices
    res = []
    for i, num in enumerate(nums):
        # Remove indices outside window
        if dq and dq[0] < i - k + 1:
            dq.popleft()
        # Maintain decreasing order in deque
        while dq and nums[dq[-1]] < num:
            dq.pop()
        dq.append(i)
        if i >= k - 1:
            res.append(nums[dq[0]])
    return res

nums = [1, 3, -1, -3, 5, 3, 6, 7]
print("Sliding Window Max (k=3):", max_sliding_window(nums, 3))


---
## 7. Algorithmic Pattern 3: Circular Queue Implementation


In [ ]:
class CircularQueue:
    def __init__(self, k: int):
        self.capacity = k
        self.queue = [0] * k
        self.head = 0
        self.tail = 0
        self.size = 0

    def enqueue(self, value: int) -> bool:
        if self.isFull(): return False
        self.queue[self.tail] = value
        self.tail = (self.tail + 1) % self.capacity
        self.size += 1
        return True

    def dequeue(self) -> bool:
        if self.isEmpty(): return False
        self.head = (self.head + 1) % self.capacity
        self.size -= 1
        return True

    def Front(self) -> int:
        return -1 if self.isEmpty() else self.queue[self.head]

    def isEmpty(self) -> bool: return self.size == 0
    def isFull(self) -> bool: return self.size == self.capacity

cq = CircularQueue(3)
cq.enqueue(1); cq.enqueue(2); cq.enqueue(3)
print("Is full?", cq.isFull())
cq.dequeue()
cq.enqueue(4)
print("Front value:", cq.Front())


---
## 8. Quick Reference Card


In [ ]:
# ==================================================================
# QUEUES – QUICK REFERENCE
# ==================================================================
from collections import deque

q = deque()
q.append(1)      # Enqueue
val = q.popleft()# Dequeue O(1)


---
## Summary

| Structure | Method | Complexity | Primary Application |
|-----------|--------|------------|---------------------|
| **`collections.deque`** | `.append()` / `.popleft()` | $O(1)$ | Optimal queue & BFS traversal |
| **Monotonic Deque** | Sliding window max | $O(n)$ | Subarray max tracking |
| **`queue.Queue`** | `.put()` / `.get()` | $O(1)$ thread-safe | Multithreaded producer-consumer |

---
*Next up: **Heaps & Priority Queues***
